# Phase 3: Data integration

In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)

churn = pd.read_csv("../data/raw/customer_churn.csv")
customers = pd.read_csv("../data/processed/customers_cleaned.csv")
engagement = pd.read_csv("../data/raw/customer_engagement_metrics.csv")
rfm = pd.read_csv("../data/raw/customer_rfm.csv")
orders = pd.read_csv("../data/processed/orders_cleaned.csv")
billing = pd.read_csv("../data/raw/subscription_billing.csv")
tickets = pd.read_csv("../data/raw/support_tickets.csv")
campaign = pd.read_csv("../data/raw/campaign_responses.csv")
pd.set_option('display.max_columns',None)
pd.set_option("display.max_rows", None)



In [2]:
datasets = {
    "customers": customers,
    "churn": churn,
    "engagement": engagement,
    "rfm": rfm,
    "orders": orders,
    "billing": billing,
    "tickets": tickets,
    "campaign": campaign
}

| Dataset    | Unique Customers | Duplicate IDs | Relationship | Action       |
| ---------- | ---------------: | ------------: | ------------ | ------------ |
| customers  |           93,551 |             0 | One-to-One   | Direct merge |
| churn      |          150,000 |        31,175 | One-to-Many  | Aggregate    |
| engagement |          150,000 |        69,237 | One-to-Many  | Aggregate    |
| rfm        |          150,000 |       467,852 | One-to-Many  | Aggregate    |
| orders     |           80,183 |        34,204 | One-to-Many  | Aggregate    |
| billing    |          126,941 |             0 | One-to-One   | Direct merge |
| tickets    |           72,253 |        26,501 | One-to-Many  | Aggregate    |
| campaign   |          146,833 |       431,562 | One-to-Many  | Aggregate    |


Before merging the datasets, customer_id relationships must be analyzed.

Some datasets contain one record per customer, while others contain multiple records per customer.

Datasets with one customer record can be directly merged.

Datasets containing multiple records per customer must first be aggregated into a single customer-level record.

After aggregation, all datasets can be merged using customer_id to create the final customer-level dataset.

In [3]:
merge_datasets = {
    "customers": customers,
    "billing": billing
}

aggregate_datasets = {
    "churn": churn,
    "engagement": engagement,
    "rfm": rfm,
    "orders": orders,
    "tickets": tickets,
    "campaign": campaign
}

In [4]:
for name,df in aggregate_datasets.items():
    print(name,": ",df['customer_id'].value_counts()[0:2])
    print("="*20)

churn :  customer_id
C000001    2
C000002    2
Name: count, dtype: int64
engagement :  customer_id
C000001    2
C000002    2
Name: count, dtype: int64
rfm :  customer_id
C000001    5
C000002    5
Name: count, dtype: int64
orders :  customer_id
C057629    7
C035280    6
Name: count, dtype: int64
tickets :  customer_id
C141628    8
C066808    6
Name: count, dtype: int64
campaign :  customer_id
C070709    16
C025342    15
Name: count, dtype: int64


In [5]:

for name,df in aggregate_datasets.items():
    print(name,": ",df[df["customer_id"]=="C000001"])
    print("="*20)

churn :         customer_id  tenure_months contract_type  monthly_spend  \
0          C000001             88      biennial         369.85   
150000     C000001             31        annual         207.86   

        days_since_last_order  support_tickets_30d  total_spend  order_count  \
0                         284                   10      29988.0           57   
150000                    398                   15      16102.8           31   

        avg_order_value payment_method  autopay  num_products_owned  \
0                999.60    credit_card        1                   8   
150000           536.76    credit_card        0                   6   

        complaint_count  nps_score  satisfaction_score  churned churn_reason  \
0                     3          5                   3        1      service   
150000                2          3                   2        1   competitor   

        predicted_churn_prob retention_offer_sent  offer_accepted  \
0                      0.66

In [6]:
churn_agg = (
    churn.sort_values("updated_at")
         .groupby("customer_id")
         .last()
         .reset_index()
)

In [7]:
engagement_agg = (
    engagement.sort_values("measured_date")
              .groupby("customer_id")
              .last()
              .reset_index()
)

In [8]:
rfm_agg = (
    rfm.groupby("customer_id")
       .agg({
           "recency_days":"mean",
           "frequency":"mean",
           "monetary":"mean",
           "r_score":"mean",
           "f_score":"mean",
           "m_score":"mean"
       })
       .reset_index()
)

In [9]:
orders_agg = (
    orders.groupby("customer_id")
          .agg({
              "order_id":"count",
              "total_amount":"sum",
              "discount_amount":"sum",
              "shipping_cost":"sum",
              "is_gift":"sum"
          })
          .reset_index()
)

In [10]:
orders_agg.rename(columns={
    "order_id":"total_orders",
    "total_amount":"total_revenue",
    "discount_amount":"total_discount",
    "shipping_cost":"total_shipping_cost",
    "is_gift":"gift_orders"
}, inplace=True)

In [11]:
tickets_agg = (
    tickets.groupby("customer_id")
           .agg({
               "ticket_id":"count",
               "resolution_time_hours":"mean"
           })
           .reset_index()
)

In [12]:

tickets_agg.rename(columns={
    "ticket_id": "total_tickets",
    "resolution_time_hours": "avg_resolution_time"
}, inplace=True)

In [13]:
campaign_agg = (
    campaign.groupby("customer_id")
            .agg({
                "campaign_id":"count",
                "responded":"sum",
                "converted":"sum",
                "revenue":"sum"
            })
            .reset_index()
)

In [14]:
campaign_agg.rename(columns={
    "campaign_id":"total_campaigns",
    "revenue":"campaign_revenue"
}, inplace=True)

In [15]:
final_df = churn_agg.copy()

In [16]:
merge_datasets = {
    "customers": customers,
    "billing": billing,
    "engagement": engagement_agg,
    "rfm": rfm_agg,
    "orders": orders_agg,
    "tickets": tickets_agg,
    "campaign": campaign_agg
}

In [17]:
for name, df in merge_datasets.items():
    final_df = final_df.merge(
        df,
        on="customer_id",
        how="left"
    )

    print(name)
    print(final_df.shape)
    print("-" * 30)

customers
(150000, 50)
------------------------------
billing
(150000, 71)
------------------------------
engagement
(150000, 92)
------------------------------
rfm
(150000, 98)
------------------------------
orders
(150000, 103)
------------------------------
tickets
(150000, 105)
------------------------------
campaign
(150000, 109)
------------------------------


In [18]:
final_df.isna().sum().sort_values(ascending=False)

total_tickets                77747
avg_resolution_time          77747
total_orders                 69817
total_shipping_cost          69817
total_revenue                69817
total_discount               69817
gift_orders                  69817
household_size               56449
signup_date                  56449
age                          56449
gender                       56449
occupation                   56449
marital_status               56449
preferred_channel            56449
preferred_language           56449
lifetime_value               56449
loyalty_points               56449
phone                        56449
account_status               56449
street_address               56449
city                         56449
postal_code                  56449
segment                      56449
name                         56449
email                        56449
country                      56449
state                        56449
last_name                    56449
first_name          

| Column                    | Meaning                       |
| ------------------------- | ----------------------------- |
| total_tickets = NaN       | Customer never raised tickets |
| avg_resolution_time = NaN | No tickets exist              |
| total_orders = NaN        | Customer never ordered        |
| total_revenue = NaN       | No purchases                  |


In [19]:
num_cols = final_df.select_dtypes(
    include=["int64", "float64"]
).columns

final_df[num_cols] = final_df[num_cols].fillna(0)

In [20]:
cat_cols = final_df.select_dtypes(
    include="object"
).columns

final_df[cat_cols] = final_df[cat_cols].fillna("Unknown")

In [21]:
final_df.isnull().sum().sum()

np.int64(0)

In [22]:
final_df.to_csv(
    "../data/processed/final_customer_data.csv",
    index=False
)